# Variational Monte Carlo (VMC)

Estimate `⟨H⟩` via sampling and use the result in a gradient step. Same `sample(O, ψ)` trace runs classical PRNG on cpu, Born-rule measurement on cpu+sim.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** estimate energy of a parameterized state. **Classical:** Metropolis-Hastings or inverse-CDF sampling. **Quantum:** projective measurement on a prepared state.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.ops import sample

O = np.array([[1.0, 0.0], [0.0, -1.0]])  # Pauli-Z
psi = np.array([1.0, 1.0]) / np.sqrt(2.0)
n_samples = 200

@trace
def prog(observable, st):
    return sample(observable, st, n_samples=n_samples)

# Pack [Re; Im] state.
state = np.concatenate([psi, np.zeros_like(psi)])
module = prog(O.tolist(), state.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
